# Example 3 — LangGraph Multi-Agent Workflow: Career Intelligence Agent

## From Single Prompts to Agent Teams

In the previous notebook, RAG gave us a way to *ground* an LLM in specific documents. But many real-world tasks are too complex for a single prompt — they require planning, information gathering, analysis, and synthesis in sequence, with each step depending on the output of the last.

Consider what it takes to answer: *"Is now a good time to pivot into ML engineering?"*

- A single LLM call gives you a generic, possibly stale answer
- A **multi-agent system** can do much better:

```
Step 1 — PLAN:     Decompose the question into concrete research sub-tasks
Step 2 — RESEARCH: Gather market data (salaries, skills, hiring trends)
Step 3 — ANALYZE:  Identify patterns and key signals from the raw data
Step 4 — WRITE:    Synthesise everything into an actionable career brief
```

Each step is handled by a **specialised agent**. They communicate through a **shared state** — a common data structure that every agent can read from and write to. **LangGraph** is the framework that wires these agents together into a controllable, observable workflow graph.

### Why Not Just Chain Prompts?

You could chain LLM calls with simple Python. But LangGraph gives you:
- **Conditional routing** — branch to different agents based on what happened
- **Cycles** — retry loops when an agent fails or returns empty results
- **State persistence** — the full conversation history travels with the workflow
- **Visual debuggability** — the graph structure makes it easy to reason about failures

## What You'll Learn

- Why multi-agent systems outperform single-agent approaches on complex tasks
- How to define specialised agents as LangGraph **nodes**
- How **shared state** flows through a directed graph
- How to add **conditional routing** to handle failures gracefully

## Before You Start

- Dependencies from `01_mcp_weather_tool.ipynb` are already installed
- Run all cells **top to bottom** in order

---
## Step 1 — Import LangGraph

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

print("LangGraph imported successfully.")

LangGraph imported successfully.


---
## Step 2 — Define the Shared State

In LangGraph, **all agents share a single state dictionary**. Think of it as the shared whiteboard the whole team writes on — each agent picks up where the previous one left off.

We use Python's `TypedDict` to declare the fields explicitly. This makes the contract between agents crystal-clear:

| Field | Set by | Contains |
|-------|--------|---------|
| `query` | You (the user) | The career role you want researched |
| `task` | Planner agent | A structured research plan |
| `data` | Researcher agent | Raw market data (salaries, skills, trends) |
| `result` | Writer agent | The polished career intelligence brief |

> **Design tip:** Good state design is half the work of building a multi-agent system. Fields that are too broad make agents hard to test; fields that are too narrow force agents to do too much parsing.

In [2]:
class WorkflowState(TypedDict):
    """The shared state that flows through every node in the graph."""
    query:  str   # The career role entered by the user
    task:   str   # Research plan generated by the Planner
    data:   str   # Market data collected by the Researcher
    result: str   # Final career brief written by the Writer

print("✅ State schema defined:", list(WorkflowState.__annotations__.keys()))

✅ State schema defined: ['query', 'task', 'data', 'result']


---
## Step 3 — Define the Three Agents

Each agent is a **plain Python function** that:
- **Receives** the current shared state dictionary
- **Returns** a dictionary of updates to merge back into the state

This is the key LangGraph contract — agents are just functions. In a production system each would call a live LLM or external API; here we use realistic simulated data so you can run everything without hitting rate limits.

| Agent | Responsibility | Real-world equivalent |
|-------|---------------|----------------------|
| **Planner** | Reads the query, produces a structured research plan | GPT-4o with a planning prompt |
| **Researcher** | Looks up salary ranges, skills, hiring trends | Job API calls (LinkedIn, Glassdoor, Levels.fyi) |
| **Writer** | Turns raw data into a polished career brief | GPT-4o with a writing/formatting prompt |

In [3]:
def planner(state: WorkflowState) -> dict:
    """
    Planner Agent — reads the user's career query and produces a
    structured research plan that guides the Researcher.

    In production: this would call an LLM with a system prompt like
    "You are a career strategist. Given a job title, return a JSON
    research plan covering salary, skills, and market trends."
    """
    query = state["query"]
    print(f"  [Planner] 📋 Career query received: '{query}'")

    task = (
        f"Research the role '{query}' across three dimensions:\n"
        f"  1. Compensation — global salary range (entry / mid / senior)\n"
        f"  2. Skills — top technical and soft skills employers require\n"
        f"  3. Market trend — is hiring growing, stable, or declining?"
    )
    print(f"  [Planner] ✅ Research plan created.")
    return {"task": task}

In [4]:
def researcher(state: WorkflowState) -> dict:
    """
    Researcher Agent — executes the research plan and returns structured
    market intelligence data for the requested role.

    In production: this would call live job-market APIs — LinkedIn Talent
    Insights, Glassdoor, Levels.fyi, or a RAG index over recent job postings.
    Here we use realistic curated data so the notebook runs without API keys.
    """
    query = state["query"]
    print(f"  [Researcher] 🔍 Gathering market data for: '{query}'")

    # Curated market intelligence — realistic 2024/2025 data
    market_db = {
        "ML Engineer": {
            "salary":  "Entry: $110k–$140k | Mid: $150k–$200k | Senior: $200k–$300k+ (US)",
            "skills":  ["Python", "PyTorch / TensorFlow", "MLOps & model serving",
                        "SQL & data pipelines", "Cloud platforms (AWS / GCP / Azure)",
                        "A/B testing & experiment design"],
            "trend":   "📈 HIGH GROWTH — ~45% YoY increase in job postings (2023→2025)",
            "hiring":  ["Google DeepMind", "Meta AI", "OpenAI", "Databricks",
                        "Stripe", "Airbnb", "Waymo"],
            "insight": ("Demand is especially strong for engineers who can bridge "
                        "research prototypes and production systems. MLOps skills "
                        "are now table-stakes at most companies above Series B."),
        },
        "Data Scientist": {
            "salary":  "Entry: $90k–$120k | Mid: $130k–$170k | Senior: $170k–$230k (US)",
            "skills":  ["Python / R", "Statistical modelling", "SQL", "Scikit-learn",
                        "Tableau / Looker", "Business storytelling"],
            "trend":   "📊 STABLE — mature role; entry-level is competitive, senior roles strong",
            "hiring":  ["Amazon", "Microsoft", "McKinsey QuantumBlack", "Netflix",
                        "Spotify", "JPMorgan", "Airbnb"],
            "insight": ("The role is bifurcating: analysts are being augmented by BI tools, "
                        "while senior data scientists who can lead cross-functional projects "
                        "and translate findings into product decisions remain highly valued."),
        },
        "AI Research Scientist": {
            "salary":  "Entry: $140k–$180k | Mid: $200k–$300k | Senior: $300k–$500k+ (US, incl. equity)",
            "skills":  ["Deep learning theory", "PyTorch (advanced)", "CUDA / GPU programming",
                        "Academic publishing", "PhD preferred", "Reinforcement learning / RLHF"],
            "trend":   "🚀 EXPLOSIVE — driven by LLM boom; extremely competitive to enter",
            "hiring":  ["Anthropic", "OpenAI", "Google DeepMind", "Meta FAIR",
                        "Microsoft Research", "Mistral AI", "Cohere"],
            "insight": ("Publishing at top venues (NeurIPS, ICML, ICLR) is still the "
                        "primary signal labs use for hiring. The compensation ceiling is "
                        "remarkably high, but fewer than 5% of PhD graduates land these roles."),
        },
        "Product Manager (AI)": {
            "salary":  "Entry: $100k–$130k | Mid: $150k–$200k | Senior: $200k–$280k (US)",
            "skills":  ["Product sense", "Technical AI literacy", "User research",
                        "SQL / analytics", "Roadmap prioritisation", "Stakeholder management"],
            "trend":   "📈 GROWING — every AI company is hiring product managers with ML literacy",
            "hiring":  ["Google", "Microsoft", "Salesforce", "HubSpot",
                        "Scale AI", "Cohere", "early-stage AI startups"],
            "insight": ("This is the fastest-growing hybrid role in tech. Companies "
                        "increasingly want PMs who understand model limitations, evaluation "
                        "frameworks, and responsible AI — not just traditional product skills."),
        },
    }

    # Look up data — fall back to ML Engineer if role isn't in our database
    role_data = market_db.get(query)
    if role_data is None:
        print(f"  [Researcher] ⚠️  '{query}' not in primary database — using closest match.")
        role_data = market_db["ML Engineer"]

    data = (
        f"ROLE: {query}\n"
        f"SALARY RANGE: {role_data['salary']}\n"
        f"TOP SKILLS: {', '.join(role_data['skills'])}\n"
        f"MARKET TREND: {role_data['trend']}\n"
        f"TOP HIRING COMPANIES: {', '.join(role_data['hiring'])}\n"
        f"KEY INSIGHT: {role_data['insight']}"
    )

    print(f"  [Researcher] ✅ Market data collected.")
    return {"data": data}

In [5]:
def writer(state: WorkflowState) -> dict:
    """
    Writer Agent — formats the raw market intelligence into a polished
    career brief that a user can act on immediately.

    In production: this would call an LLM with a formatting prompt, potentially
    personalising the brief based on the user's current CV or skill profile.
    """
    data  = state["data"]
    query = state["query"]
    print(f"  [Writer] ✍️  Drafting career brief for: '{query}'")

    result = (
        f"╔══════════════════════════════════════════════════════════╗\n"
        f"  CAREER INTELLIGENCE BRIEF\n"
        f"  Role: {query.upper()}\n"
        f"╚══════════════════════════════════════════════════════════╝\n\n"
        f"{data}\n\n"
        f"━━━ BOTTOM LINE FOR CANDIDATES ━━━\n"
        f"If you are a grad student or early-career professional considering\n"
        f"this path: focus on building a portfolio of *deployed* projects\n"
        f"(not just notebooks), contribute to open-source ML repos, and\n"
        f"target companies in the 'Top Hiring' list for internships or\n"
        f"new-grad roles. Networking with alumni at these firms accelerates\n"
        f"the process more than any single certification.\n\n"
        f"─── Report generated by Career Intelligence Agent v1.0 ───"
    )

    print(f"  [Writer] ✅ Career brief ready.")
    return {"result": result}

---
## Step 4 — Build the Workflow Graph

Now we wire the agents together. Each becomes a **node**; the connections are **edges**. The graph is then **compiled** into a runnable application.

```
                ┌───────────┐
   START ──────►│  planner  │
                └─────┬─────┘
                      │
                      ▼
               ┌────────────┐
               │ researcher │
               └─────┬──────┘
                     │
                     ▼
                ┌────────┐
                │ writer │
                └───┬────┘
                    │
                   END
```

Notice how clean this is: the graph *structure* is separate from the agent *logic*. You can swap out any agent without touching the wiring, or add a new agent (e.g., a `validator`) by inserting a node.

In [6]:
# Create the graph with our typed state schema
graph = StateGraph(WorkflowState)

# Register each agent as a named node
graph.add_node("planner",    planner)
graph.add_node("researcher", researcher)
graph.add_node("writer",     writer)

# Define execution order
graph.set_entry_point("planner")             # Always starts here
graph.add_edge("planner",    "researcher")   # Planner → Researcher
graph.add_edge("researcher", "writer")       # Researcher → Writer
graph.add_edge("writer",     END)            # Writer → Done

# Compile into a runnable app
app = graph.compile()

print("✅ Graph compiled.")
print("   Execution order: planner → researcher → writer → END")

✅ Graph compiled.
   Execution order: planner → researcher → writer → END


---
## Step 5 — Run the Workflow

We kick off the graph by providing only the initial user query. Watch how the shared state gets populated step by step — each agent writes its output and passes the enriched state downstream.

Try different roles: `"ML Engineer"`, `"Data Scientist"`, `"AI Research Scientist"`, `"Product Manager (AI)"`

In [7]:
print("Starting Career Intelligence workflow...")
print("=" * 55)

# ✏️  Change this to any role you want to research:
role = "ML Engineer"   # Try: "Data Scientist" | "AI Research Scientist" | "Product Manager (AI)"

final_state = app.invoke({"query": role})

print("=" * 55)
print("\n📊 Final state snapshot:")
for key, value in final_state.items():
    preview = repr(value)[:80] + "..." if len(repr(value)) > 80 else repr(value)
    print(f"  {key:8s} → {preview}")

print("\n")
print(final_state["result"])

Starting Career Intelligence workflow...
  [Planner] 📋 Career query received: 'ML Engineer'
  [Planner] ✅ Research plan created.
  [Researcher] 🔍 Gathering market data for: 'ML Engineer'
  [Researcher] ✅ Market data collected.
  [Writer] ✍️  Drafting career brief for: 'ML Engineer'
  [Writer] ✅ Career brief ready.

📊 Final state snapshot:
  query    → 'ML Engineer'
  task     → "Research the role 'ML Engineer' across three dimensions:\n  1. Compensation — g...
  data     → 'ROLE: ML Engineer\nSALARY RANGE: Entry: $110k–$140k | Mid: $150k–$200k | Senior...
  result   → "╔══════════════════════════════════════════════════════════╗\n  CAREER INTELLIG...


╔══════════════════════════════════════════════════════════╗
  CAREER INTELLIGENCE BRIEF
  Role: ML ENGINEER
╚══════════════════════════════════════════════════════════╝

ROLE: ML Engineer
SALARY RANGE: Entry: $110k–$140k | Mid: $150k–$200k | Senior: $200k–$300k+ (US)
TOP SKILLS: Python, PyTorch / TensorFlow, MLOps & model serving, SQL &

---
## Step 6 — Conditional Routing: Handling Failure Gracefully

Real workflows break. APIs time out, databases return empty results, external services go down. A robust agent system needs to **detect failures and retry** rather than silently producing a broken output.

LangGraph's `add_conditional_edges` lets you route to a *different node* based on the current state. Here we simulate a common real-world scenario: querying a niche or emerging role that isn't in our primary database. The workflow detects the empty result and automatically falls back to a broader search strategy.

```
             ┌───────────────┐
START ──────►│    planner    │
             └───────┬───────┘
                     │
                     ▼
             ┌───────────────┐
        ┌───►│ researcher_v2 │
        │    └───────┬───────┘
        │            │
        │     ┌──────▼──────────────────┐
        │     │ route_after_research()  │  ← decision function
        │     └──────┬──────────────────┘
        │            │
        │    data empty?      data found?
        └────────────┘            │
         (retry)                  ▼
                            ┌─────────┐
                            │ writer  │──► END
                            └─────────┘
```

In [8]:
def researcher_v2(state: dict) -> dict:
    """
    Researcher v2 — simulates a two-stage fallback search strategy.

    Attempt 1: Queries the primary specialised database. Returns empty
               for niche / emerging roles not yet in the database.
    Attempt 2: Falls back to a broader industry-wide search that covers
               any AI-adjacent role with generalised data.
    """
    query   = state["query"]
    attempt = state.get("attempt", 0) + 1
    print(f"  [Researcher v2] 🔍 Attempt #{attempt} — searching for: '{query}'")

    if attempt < 2:
        # Simulated: first pass hits an empty cache for an emerging role
        print("  [Researcher v2] ⚠️  Primary database: no match found. Queuing retry...")
        return {"data": None, "attempt": attempt}   # None is unambiguously "no data"
    else:
        # Simulated: broader fallback search succeeds
        data = (
            f"ROLE: {query}\n"
            f"SALARY RANGE: $120k–$200k (estimated; limited data for emerging roles)\n"
            f"TOP SKILLS: Python, ML frameworks, system design, cloud infrastructure\n"
            f"MARKET TREND: 📈 Emerging role — growing rapidly; limited historical benchmarks\n"
            f"TOP HIRING COMPANIES: Well-funded AI startups, Big Tech R&D labs, AI consultancies\n"
            f"KEY INSIGHT: This is a fast-moving space with high variance in role definitions.\n"
            f"   Target companies that have published work in this area and read their job\n"
            f"   descriptions carefully — skills requirements shift month to month."
        )
        print(f"  [Researcher v2] ✅ Fallback search succeeded on attempt #{attempt}.")
        return {"data": data, "attempt": attempt}


def route_after_research(state: dict) -> str:
    """
    Router — inspects the state after researcher_v2 runs and decides
    whether to proceed to the writer or loop back for a retry.
    """
    if state.get("data"):                            # truthy non-empty string → proceed
        print("  [Router] ✅ Data found — proceeding to writer.")
        return "writer"
    else:                                            # None or empty string → retry
        print("  [Router] 🔁 No data — retrying researcher.")
        return "researcher_v2"


# ── State schema for the retry workflow ────────────────────────────────────
# We define a FRESH TypedDict (not inheriting from WorkflowState) with
# total=False so all fields are optional. This is the safest pattern with
# LangGraph: it allows the graph to be invoked with a partial initial state
# and lets nodes return partial updates without schema validation errors.

class WorkflowStateV2(TypedDict, total=False):
    query:   str   # career role entered by the user
    task:    str   # research plan from planner
    data:    str   # market data from researcher (None if not yet found)
    result:  str   # final brief from writer
    attempt: int   # retry counter — not present in WorkflowState

# ── Build graph v2 ──────────────────────────────────────────────────────────
graph_v2 = StateGraph(WorkflowStateV2)

graph_v2.add_node("planner",       planner)
graph_v2.add_node("researcher_v2", researcher_v2)
graph_v2.add_node("writer",        writer)

graph_v2.set_entry_point("planner")
graph_v2.add_edge("planner", "researcher_v2")

# Conditional edge: call route_after_research after researcher_v2 to decide next step
graph_v2.add_conditional_edges(
    "researcher_v2",
    route_after_research,
    {
        "writer":        "writer",         # data found → go to writer
        "researcher_v2": "researcher_v2",  # no data   → loop back
    }
)
graph_v2.add_edge("writer", END)

app_v2 = graph_v2.compile()

# ── Run ─────────────────────────────────────────────────────────────────────
print("Running with retry logic — querying a niche role not in the primary DB...")
print("=" * 55)
result_v2 = app_v2.invoke({"query": "AI Safety Researcher"})
print("=" * 55)
print()
print(result_v2["result"])

Running with retry logic — querying a niche role not in the primary DB...
  [Planner] 📋 Career query received: 'AI Safety Researcher'
  [Planner] ✅ Research plan created.
  [Researcher v2] 🔍 Attempt #1 — searching for: 'AI Safety Researcher'
  [Researcher v2] ⚠️  Primary database: no match found. Queuing retry...
  [Router] 🔁 No data — retrying researcher.
  [Researcher v2] 🔍 Attempt #2 — searching for: 'AI Safety Researcher'
  [Researcher v2] ✅ Fallback search succeeded on attempt #2.
  [Router] ✅ Data found — proceeding to writer.
  [Writer] ✍️  Drafting career brief for: 'AI Safety Researcher'
  [Writer] ✅ Career brief ready.

╔══════════════════════════════════════════════════════════╗
  CAREER INTELLIGENCE BRIEF
  Role: AI SAFETY RESEARCHER
╚══════════════════════════════════════════════════════════╝

ROLE: AI Safety Researcher
SALARY RANGE: $120k–$200k (estimated; limited data for emerging roles)
TOP SKILLS: Python, ML frameworks, system design, cloud infrastructure
MARKET TREND:

---
## Try It Yourself — Challenges

**Challenge 1 — Query all four roles and compare**

Run Step 5 four times with: `"ML Engineer"`, `"Data Scientist"`, `"AI Research Scientist"`, `"Product Manager (AI)"`. Which role would you target given your background? Which surprised you most?

**Challenge 2 — Add a fourth agent: the Critic**

Insert a `critic` agent between `researcher` and `writer` that reads the raw data and flags any potential concerns:
```python
def critic(state: WorkflowState) -> dict:
    data = state["data"]
    # Add your logic: e.g., warn if trend is "stable" or "declining"
    critique = "⚠️ Highly competitive entry level — portfolio projects are essential."
    return {"data": data + f"\nCRITIQUE: {critique}"}
```
Then add it to the graph: `planner → researcher → critic → writer → END`

**Challenge 3 — Connect to a live LLM**

Replace the hardcoded `task` string in the `planner` with a real OpenAI call:
```python
from openai import OpenAI
client = OpenAI()

def planner(state):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"Create a research plan for the role: {state['query']}"}]
    )
    return {"task": response.choices[0].message.content}
```
Notice how the graph structure doesn't change at all — just the agent internals.

**Challenge 4 — Add a max-retry safeguard**

The current retry loop could run forever if the data is always empty. Modify `researcher_v2` and `route_after_research` to stop after 3 attempts and write a fallback message to the state.

---
## Summary

You just built a complete stateful multi-agent workflow with both linear and conditional execution:

| Concept | What you saw |
|---------|-------------|
| **Nodes** | Python functions that read state and return updates |
| **Edges** | Define the execution sequence between agents |
| **Shared state** | A typed dictionary that carries all data through the entire workflow |
| **Conditional edges** | Dynamic routing based on what actually happened at runtime |
| **Retry loops** | Cycles in the graph that let agents recover from failures |

### The Big Picture

```
Single LLM call   →  good for simple Q&A
RAG               →  grounds answers in your documents
Multi-agent graph →  handles complex, multi-step tasks with failure recovery
```

Each layer builds on the last. The notebooks in this workshop have taken you through all three.

### What This Looks Like in Production

In a real system, each agent would:
- Call a fine-tuned or instruction-following LLM with a specialised system prompt
- Have access to specific tools (database queries, API calls, file system operations)
- Write structured outputs (JSON, not plain strings) that downstream agents can parse reliably
- Be monitored with tracing tools like LangSmith or Langfuse to debug failures

The *graph structure* you defined here is exactly what production agentic systems use — the only difference is the richness of what happens inside each node.